# 2a. SCOPe remote homology

**Paper:** SCOPe remote homology (AUROC1 / Table of Family, Superfamily, Fold sensitivity). Figures are drawn in `3_figures.ipynb`.

All-vs-all search (`query = target`) on each method DB, then sensitivity up to the first wrong-fold hit.

**Two protocols — do not treat as one metric:**

| Methods | Search | Sensitivity |
|---------|--------|-------------|
| Foldseek / predicted 3Di | `easy-search -s 9.5 --max-seqs 2000 -e 10` | **hitlist**: denominator = same-level hits in the TSV (aligned with `new_scope40`) |
| MMseqs2 | `search -a -s 7.5 --max-seqs 2000 -e 10000` then `convertalis` | **catalog**: `bench.noselfhit.awk` — denominator = all catalog homologs; valid queries only (family>1 AND remote sfam AND remote fold); zero hits → 0 |

Shared rules: skip self-hit; first wrong fold is FP; Family / Superfamily / Fold are mutually exclusive; AUC = mean query sensitivity.

| | Path |
|--|------|
| Input | `work/DB/*_DB/`, `bin/{foldseek,mmseqs}`, `work/labels/scop_lookup.tsv` |
| Output | `work/aln/{method}_easy.tsv` (+ `.meta.json` param fingerprint) |
| | `work/metrics/{method}_easy_{fam,sup,fol}.tsv` |
| | `work/metrics/auc_easy.csv` |

**Node: CPU compute node, 64 threads.** Login node: lower `THREADS`. Default is **serial** (one method after another). Wall-clock is hours.

**Next:** `3_figures.ipynb`. Translation accuracy is independent: `2b_translation_accuracy.ipynb`.

### Appendix (not the default path)

To search methods in parallel, submit separate jobs with `ONLY_METHODS=["foldseek"]` (etc.). Keep the serial path for the paper reproduction.


## Environment


In [ ]:
NOTEBOOK_NAME = "2a_remote_homology.ipynb"

import os
import platform
import subprocess
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if not (cwd / NOTEBOOK_NAME).is_file():
    raise SystemExit(
        f"Start this notebook from the project root (cwd must contain {NOTEBOOK_NAME}). "
        f"Current cwd: {cwd}"
    )

CONDA_ENV = "ESM3_3Di_5090"
print("notebook:", NOTEBOOK_NAME)
print("cwd:", cwd)
print("python:", sys.executable)
print("version:", sys.version.split()[0])
print("platform:", platform.platform())
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV", "(unset)"))

if CONDA_ENV not in sys.executable:
    expected = Path.home() / ".conda" / "envs" / CONDA_ENV / "bin" / "python"
    raise SystemExit(
        f"Kernel is not {CONDA_ENV} (current: {sys.executable}). "
        f"Select kernel {CONDA_ENV} and Restart. Do not pip into miniforge3 python3.12. "
        f"Expected: {expected}"
    )


def _bin_version(name: str) -> str:
    path = cwd / "bin" / name
    if not path.is_file():
        return "(not installed yet; run 0_prepare_scope40.ipynb)"
    try:
        proc = subprocess.run([str(path), "version"], capture_output=True, text=True, check=False)
        lines = (proc.stdout or proc.stderr or "").strip().splitlines()
        return lines[0] if lines else "(unknown)"
    except OSError as exc:
        return f"(failed: {exc})"


print("foldseek:", _bin_version("foldseek"))
print("mmseqs:", _bin_version("mmseqs"))


## Configuration

本格在五本 notebook 中**字节级相同**。改方法表、搜索参数或 URL 时：只改 `0_prepare_scope40.ipynb` 这一格，再整格复制到另外四本。发布前可用 checksum 核对五本是否一致。


In [ ]:
# =============================================================================
# Configuration — copy this entire cell into all five notebooks.
# Change methods or search parameters here in 0_prepare_scope40.ipynb, then
# paste the same cell into 1_build / 2a / 2b / 3_figures.
# =============================================================================
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
HOME = ROOT.parent

CONDA_ENV = "ESM3_3Di_5090"
FOLDSEEK_VERSION = "10-941cd33"
MMSEQS_VERSION = "18-8cc5c"

TEMP = ROOT / "tmp"
WORK_DIR = ROOT / "work"
BIN_DIR = ROOT / "bin"
WORK_TMP_DIR = WORK_DIR / "tmp"

GT_FASTA_DIR = WORK_DIR / "GT_fasta"
AA_FASTA = GT_FASTA_DIR / "DB_aa.fasta"
GT_DI_FASTA = GT_FASTA_DIR / "DB_di.fasta"

AA2DI_FASTA_DIR = WORK_DIR / "aa2di_fasta"
DI2AA_FASTA_DIR = WORK_DIR / "di2aa_fasta"

DBS_DIR = WORK_DIR / "DB"
FOLDSEEK_GT_DIR = DBS_DIR / "foldseek_DB"
MMSEQS_GT_DIR = DBS_DIR / "mmseqs_DB"

LABEL_DIR = WORK_DIR / "labels"
SCOP_LOOKUP = LABEL_DIR / "scop_lookup.tsv"
LEGACY_LABEL_DIR = WORK_DIR / "lable"

ALN_DIR = WORK_DIR / "aln"
METRICS_DIR = WORK_DIR / "metrics"
FIGURES_DIR = WORK_DIR / "figures"
TRANSLATION_METRICS_DIR = METRICS_DIR / "translation"
WORK_BUNDLE = WORK_DIR / "scope40_work_bundle.tar.gz"

FOLDSEEK_BIN = BIN_DIR / "foldseek"
MMSEQS_BIN = BIN_DIR / "mmseqs"

FOLDSEEK_URL = (
    "https://github.com/steineggerlab/foldseek/releases/download/"
    f"{FOLDSEEK_VERSION}/foldseek-linux-avx2.tar.gz"
)
MMSEQS_URL = (
    "https://github.com/soedinglab/MMseqs2/releases/download/"
    f"{MMSEQS_VERSION}/mmseqs-linux-avx2.tar.gz"
)
FOLDSEEK_TMP_DIR = TEMP / "foldseek"
MMSEQS_TMP_DIR = TEMP / "mmseqs"
FOLDSEEK_TARBALL = TEMP / "foldseek-linux-avx2.tar.gz"
MMSEQS_TARBALL = TEMP / "mmseqs-linux-avx2.tar.gz"

SCOP_CLA_NAME = "dir.cla.scope.2.08-stable.txt"
SCOP_DES_NAME = "dir.des.scope.2.08-stable.txt"
SOURCE_ARCHIVE_NAME = "pdbstyle-sel-gs-bib-40-2.08.tgz"
SCOP_CLA_FALLBACK = HOME / "SCOPE" / SCOP_CLA_NAME
HF_BASE = "https://huggingface.co/datasets/caijihuize/scope40_pdbstyle/resolve/main"

# Foldseek / predicted 3Di: aligned with new_scope40 easy-search
EASY_SEARCH_PARAMS = {
    "sensitivity": 9.5,
    "max_seqs": 2000,
    "evalue": 10.0,
    "threads": 64,
}
# MMseqs2: aligned with foldseek-analysis/scopbenchmark/scripts/runMMseqs.sh
MMSEQS_SEARCH_PARAMS = {
    "sensitivity": 7.5,
    "max_seqs": 2000,
    "evalue": 10000,
    "threads": 64,
    "add_backtrace": True,
}
PREPARE_THREADS = 16
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# Homology search methods. protocol must not be mixed as one AUC.
METHODS: list[dict] = [
    {"name": "Foldseek (AA+3Di)", "key": "foldseek", "engine": "foldseek", "aa2di": None, "protocol": "hitlist"},
    {"name": "MMseqs2", "key": "mmseqs", "engine": "mmseqs", "aa2di": None, "protocol": "catalog"},
    {"name": "ESM3-3Di", "key": "ESM3", "engine": "foldseek", "aa2di": "DB_ESM3_aa2di.fasta", "protocol": "hitlist"},
    {"name": "ESM3-LoRA", "key": "ESM3_LoRA", "engine": "foldseek", "aa2di": "DB_ESM3_LoRA_aa2di.fasta", "protocol": "hitlist"},
    {"name": "ProstT5 (translate)", "key": "ProstT5", "engine": "foldseek", "aa2di": "DB_ProstT5_translate_aa2di.fasta", "protocol": "hitlist"},
    {"name": "SaProt", "key": "SaProt", "engine": "foldseek", "aa2di": "DB_SaProt_aa2di.fasta", "protocol": "hitlist"},
]
# Translation accuracy (bidirectional). Homology search uses aa2di only.
TRANSLATION_METHODS: list[dict] = [
    {"name": "ESM3-3Di", "key": "ESM3", "aa2di": "DB_ESM3_aa2di.fasta", "di2aa": "DB_ESM3_di2aa.fasta"},
    {"name": "ESM3-LoRA", "key": "ESM3_LoRA", "aa2di": "DB_ESM3_LoRA_aa2di.fasta", "di2aa": "DB_ESM3_LoRA_di2aa.fasta"},
    {"name": "ProstT5 (translate)", "key": "ProstT5", "aa2di": "DB_ProstT5_translate_aa2di.fasta", "di2aa": "DB_ProstT5_translate_di2aa.fasta"},
    {"name": "SaProt", "key": "SaProt", "aa2di": "DB_SaProt_aa2di.fasta", "di2aa": "DB_SaProt_di2aa.fasta"},
]
PALETTE = {
    "Foldseek (AA+3Di)": "#2b5c8f",
    "MMseqs2": "#666666",
    "ESM3-3Di": "#d95f02",
    "ESM3-LoRA": "#1b9e77",
    "ProstT5 (translate)": "#7570b3",
    "SaProt": "#e7298a",
}


def run_cmd(argv: list[str]) -> None:
    """Print then run an external command. Logs are supplementary material."""
    print("[CMD]", " ".join(str(x) for x in argv), flush=True)
    subprocess.run([str(x) for x in argv], check=True)


def require_file(path: Path, hint: str) -> Path:
    """Fail with a pointer to the upstream notebook if a required file is missing."""
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}\n{hint}")
    return path


def meta_path(output: Path) -> Path:
    return output.with_name(output.name + ".meta.json")


def skip_if_exists(output: Path, payload: dict | None = None, skip_existing: bool = True) -> bool:
    """Skip when output exists. If payload is given, require a matching .meta.json.

    MMseqs TSV without a fingerprint is treated as stale (catalog protocol change).
    Other outputs without a fingerprint are kept; set SKIP_EXISTING=False to force.
    """
    if not skip_existing:
        return False
    if not output.is_file() or output.stat().st_size == 0:
        return False
    if payload is None:
        return True
    meta = meta_path(output)
    if not meta.is_file():
        if payload.get("engine") == "mmseqs":
            print(f"[rerun] {output.name}: no fingerprint; MMseqs catalog params need a fresh search")
            return False
        print(f"[warn] {output.name}: no fingerprint; keeping existing file (set SKIP_EXISTING=False to re-run)")
        return True
    try:
        stored = json.loads(meta.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return False
    if stored != payload:
        print(f"[rerun] {output.name}: Configuration changed")
        return False
    return True


def write_meta(output: Path, payload: dict) -> None:
    meta_path(output).write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def search_params_payload(engine: str, threads: int | None = None) -> dict:
    params = dict(MMSEQS_SEARCH_PARAMS if engine == "mmseqs" else EASY_SEARCH_PARAMS)
    params["engine"] = engine
    if threads is not None:
        params["threads"] = int(threads)
    return params


def method_by_key(method_key: str) -> dict:
    for row in METHODS:
        if row["key"] == method_key:
            return row
    raise KeyError(f"Unknown method_key: {method_key}")


def predicted_methods() -> list[dict]:
    return [row for row in METHODS if row["aa2di"] is not None]


def db_prefix(method_key: str) -> Path:
    return DBS_DIR / f"{method_key}_DB" / "DB"


def aln_tsv(method_key: str) -> Path:
    return ALN_DIR / f"{method_key}_easy.tsv"


def aln_tmp_dir(method_key: str) -> Path:
    return WORK_TMP_DIR / f"easy_{method_key}"


def metric_prefix(method_key: str) -> Path:
    return METRICS_DIR / f"{method_key}_easy"


def translation_per_seq_path(task: str, method_key: str) -> Path:
    return TRANSLATION_METRICS_DIR / f"{task}_{method_key}_per_seq.tsv"


def translation_summary_path(task: str) -> Path:
    return TRANSLATION_METRICS_DIR / f"{task}_summary.csv"


def scop_cla_path() -> Path:
    for path in (TEMP / SCOP_CLA_NAME, SCOP_CLA_FALLBACK):
        if path.is_file():
            return path
    return TEMP / SCOP_CLA_NAME


def work_ready() -> bool:
    return (
        (FOLDSEEK_GT_DIR / "DB").is_file()
        and (MMSEQS_GT_DIR / "DB").is_file()
        and AA_FASTA.is_file()
        and GT_DI_FASTA.is_file()
        and SCOP_LOOKUP.is_file()
    )


def ensure_work_dirs() -> None:
    for directory in (
        TEMP, BIN_DIR, GT_FASTA_DIR, AA2DI_FASTA_DIR, DI2AA_FASTA_DIR, DBS_DIR,
        LABEL_DIR, ALN_DIR, METRICS_DIR, TRANSLATION_METRICS_DIR, FIGURES_DIR, WORK_TMP_DIR,
    ):
        directory.mkdir(parents=True, exist_ok=True)
    legacy = LEGACY_LABEL_DIR / "scop_lookup.tsv"
    if not SCOP_LOOKUP.is_file() and legacy.is_file():
        shutil.copy2(legacy, SCOP_LOOKUP)
        print(f"[ok] migrated {legacy} -> {SCOP_LOOKUP}")


def cleanup_tmp(*, also_work_tmp: bool = True) -> None:
    """Remove tmp/ and work/tmp/ only. Keep work/ products and bin/."""
    targets = [TEMP]
    if also_work_tmp:
        targets.append(WORK_TMP_DIR)
    for path in targets:
        if path.exists():
            shutil.rmtree(path)
            print(f"[ok] cleaned {path}")
        else:
            print(f"[skip] {path} (absent)")


print("ROOT:", ROOT)
print("conda env:", CONDA_ENV)
print("Foldseek:", FOLDSEEK_VERSION, "MMseqs:", MMSEQS_VERSION)
print("METHODS:", [(m["key"], m["engine"], m["protocol"]) for m in METHODS])


## Run flags


In [ ]:
SKIP_EXISTING = True
THREADS = int(EASY_SEARCH_PARAMS["threads"])  # 64 on a compute node; 8 or 16 on login
ONLY_METHODS = None  # e.g. ["foldseek", "mmseqs", "ESM3_LoRA"]; None = all

ensure_work_dirs()
print("foldseek:", FOLDSEEK_BIN)
print("mmseqs:", MMSEQS_BIN)
print("DB:", DBS_DIR)
print("lookup:", SCOP_LOOKUP)
print("methods:", [(m["key"], m["engine"], m["protocol"]) for m in METHODS])
print("Foldseek/pred params:", EASY_SEARCH_PARAMS)
print("MMseqs params:", MMSEQS_SEARCH_PARAMS)
print(f"SKIP_EXISTING={SKIP_EXISTING}  THREADS={THREADS}  ONLY_METHODS={ONLY_METHODS}")


## Helpers


In [ ]:
import gc
import re
import traceback
from collections import Counter, defaultdict

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


def selected_methods() -> list[dict]:
    if ONLY_METHODS is None:
        return list(METHODS)
    return [method_by_key(key) for key in ONLY_METHODS]


print("helpers ready")


## Step A — Search

Write `work/aln/{key}_easy.tsv`. Skip uses `{tsv}.meta.json`; if Configuration search params change, the search is re-run. An MMseqs TSV with no fingerprint is treated as stale.


In [ ]:
def foldseek_easy_search(
    query_db: Path,
    output_tsv: Path,
    tmp_dir: Path,
    target_db: Path | None = None,
    threads: int | None = None,
    skip_existing: bool = True,
) -> Path:
    target_db = Path(target_db or query_db)
    threads = int(threads if threads is not None else EASY_SEARCH_PARAMS["threads"])
    payload = search_params_payload("foldseek", threads)
    if skip_if_exists(output_tsv, payload, skip_existing):
        print(f"[skip] {output_tsv} ({output_tsv.stat().st_size} bytes)")
        return output_tsv

    require_file(FOLDSEEK_BIN, hint="Run 0_prepare_scope40.ipynb.")
    require_file(query_db, hint="Run 1_build_predicted_dbs.ipynb for predicted methods.")
    require_file(target_db, hint="Run 1_build_predicted_dbs.ipynb for predicted methods.")
    output_tsv.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir.mkdir(parents=True, exist_ok=True)
    run_cmd([
        str(FOLDSEEK_BIN), "easy-search",
        str(query_db), str(target_db), str(output_tsv), str(tmp_dir),
        "--threads", str(threads),
        "-s", str(EASY_SEARCH_PARAMS["sensitivity"]),
        "--max-seqs", str(EASY_SEARCH_PARAMS["max_seqs"]),
        "-e", str(EASY_SEARCH_PARAMS["evalue"]),
    ])
    require_file(output_tsv, hint="foldseek easy-search did not write the TSV.")
    write_meta(output_tsv, payload)
    print(f"[ok] {output_tsv}")
    return output_tsv


def mmseqs_search(
    query_db: Path,
    output_tsv: Path,
    tmp_dir: Path,
    target_db: Path | None = None,
    threads: int | None = None,
    skip_existing: bool = True,
) -> Path:
    """search + convertalis. easy-search accepts FASTA only, not an existing DB."""
    target_db = Path(target_db or query_db)
    threads = int(threads if threads is not None else MMSEQS_SEARCH_PARAMS["threads"])
    params = MMSEQS_SEARCH_PARAMS
    payload = search_params_payload("mmseqs", threads)
    if skip_if_exists(output_tsv, payload, skip_existing):
        print(f"[skip] {output_tsv} ({output_tsv.stat().st_size} bytes)")
        return output_tsv

    require_file(MMSEQS_BIN, hint="Run 0_prepare_scope40.ipynb.")
    require_file(query_db, hint="Run 0_prepare_scope40.ipynb.")
    require_file(target_db, hint="Run 0_prepare_scope40.ipynb.")
    output_tsv.parent.mkdir(parents=True, exist_ok=True)
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)
    search_tmp = tmp_dir / "search_tmp"
    result_db = tmp_dir / "result"
    search_tmp.mkdir(parents=True, exist_ok=True)

    cmd_search = [
        str(MMSEQS_BIN), "search",
        str(query_db), str(target_db), str(result_db), str(search_tmp),
        "--threads", str(threads),
        "-s", str(params["sensitivity"]),
        "--max-seqs", str(params["max_seqs"]),
        "-e", str(params["evalue"]),
    ]
    if params.get("add_backtrace"):
        cmd_search.append("-a")
    run_cmd(cmd_search)
    run_cmd([
        str(MMSEQS_BIN), "convertalis",
        str(query_db), str(target_db), str(result_db), str(output_tsv),
        "--threads", str(threads),
        "--format-output", "query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits",
    ])
    require_file(output_tsv, hint="mmseqs convertalis did not write the TSV.")
    if output_tsv.stat().st_size == 0:
        raise FileNotFoundError(f"Empty alignment TSV: {output_tsv}")
    write_meta(output_tsv, payload)
    print(f"[ok] {output_tsv}")
    return output_tsv


def search_one(method_key: str, skip_existing: bool = True, threads: int | None = None) -> Path:
    method = method_by_key(method_key)
    kwargs = dict(
        query_db=db_prefix(method_key),
        output_tsv=aln_tsv(method_key),
        tmp_dir=aln_tmp_dir(method_key),
        skip_existing=skip_existing,
        threads=threads,
    )
    if method["engine"] == "mmseqs":
        return mmseqs_search(**kwargs)
    if method["engine"] == "foldseek":
        return foldseek_easy_search(**kwargs)
    raise ValueError(f"Unknown engine: {method['engine']} ({method_key})")


aln_paths: dict[str, Path] = {}
for method in selected_methods():
    print(f"\n=== search: {method['key']} ({method['engine']}, {method['protocol']}) ===")
    aln_paths[method["key"]] = search_one(
        method["key"], skip_existing=SKIP_EXISTING, threads=THREADS,
    )

for key, path in aln_paths.items():
    nlines = sum(1 for _ in path.open()) if path.is_file() else 0
    print(f"{key:12s} -> {path.name}  lines={nlines:,}")


## Step B — Evaluate

- **hitlist** (`calc_fp_rates`): denominator = same-level hits inside the alignment TSV.
- **catalog** (`calc_fp_rates_catalog`): foldseek-analysis `bench.noselfhit.awk`; denominator = all homologs in the SCOPe catalog; only queries that have family members, remote superfamily members, and remote fold members; missing hits count as FN (sensitivity 0).

Lookup: `work/labels/scop_lookup.tsv`. Catalog metrics are always recomputed (cheap). Hitlist metrics skip when `*_fam.tsv` already exists.


In [ ]:
def remove_family_number(scop_class: str) -> str:
    return re.sub(r"\.[0-9]+$", "", scop_class)


def resolve_scop_class(qid: str, id2cls: dict[str, str]) -> str | None:
    candidates: list[str] = [qid]
    base_model = re.sub(r"_MODEL_.*", "", qid)
    if base_model != qid:
        candidates.append(base_model)
    for cand in list(candidates):
        if "." in cand:
            base_chain = re.sub(r"_[A-Za-z0-9]+$", "", cand)
            if base_chain != cand:
                candidates.append(base_chain)
    for cand in candidates:
        if cand in id2cls:
            return id2cls[cand]
    return None


def build_scop_lookup(skip_existing: bool = True) -> Path:
    if skip_existing and SCOP_LOOKUP.is_file() and SCOP_LOOKUP.stat().st_size > 0:
        print(f"[skip] SCOP lookup exists: {SCOP_LOOKUP}")
        return SCOP_LOOKUP
    scop_cla = scop_cla_path()
    require_file(scop_cla, hint="Need dir.cla (from 0_prepare tmp, or ~/SCOPE/).")
    ensure_work_dirs()
    id2cls: dict[str, str] = {}
    with scop_cla.open() as handle:
        for line in handle:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.strip().split("\t")
            if len(parts) >= 4:
                id2cls[parts[0].strip()] = parts[3].strip()
    all_ids: set[str] = set()
    with AA_FASTA.open() as handle:
        for line in handle:
            if line.startswith(">"):
                qid = line[1:].strip().split()[0]
                if qid:
                    all_ids.add(qid)
    missed = 0
    with SCOP_LOOKUP.open("w") as out:
        for qid in sorted(all_ids):
            cls = resolve_scop_class(qid, id2cls)
            if cls is None:
                missed += 1
                continue
            out.write(f"{qid}\t{cls}\n")
    print(f"[ok] {SCOP_LOOKUP}  matched={len(all_ids) - missed} unmatched={missed}")
    return SCOP_LOOKUP


def load_scop_levels() -> pd.DataFrame:
    rows = []
    with SCOP_LOOKUP.open() as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) < 2:
                continue
            fam = parts[1].strip()
            sf = remove_family_number(fam)
            fo = remove_family_number(sf)
            rows.append({"id": parts[0].strip(), "fa": fam, "sf": sf, "fo": fo})
    return pd.DataFrame(rows)


def calc_fp_rates(aln_tsv_path: Path, cla: pd.DataFrame, out_prefix: Path) -> dict[str, Path]:
    """Hitlist protocol: denominator = same-level hits in the TSV."""
    print(f"  hitlist: {aln_tsv_path}", flush=True)
    aln = pd.read_csv(aln_tsv_path, sep="\t", header=None, usecols=[0, 1], names=["qid", "tid"], dtype=str)
    print(f"  rows: {len(aln):,}", flush=True)
    aln = aln[aln["qid"] != aln["tid"]].copy()
    print(f"  after self-hit: {len(aln):,}", flush=True)

    work = aln.merge(cla, left_on="qid", right_on="id", how="inner")
    work = work.rename(columns={"fo": "qfo", "sf": "qsf", "fa": "qfa"}).drop(columns=["id"])
    work = work.merge(cla, left_on="tid", right_on="id", how="left")
    work = work.rename(columns={"fo": "tfo", "sf": "tsf", "fa": "tfa"}).drop(columns=["id"])

    is_wrong_fold = (work["qfo"] != work["tfo"]).fillna(True)
    work["seen_fp"] = is_wrong_fold.groupby(work["qid"], sort=False).cumsum().gt(0).astype("int8")

    same_fo = work["qfo"] == work["tfo"]
    same_sf = work["qsf"] == work["tsf"]
    same_fa = work["qfa"] == work["tfa"]
    count_fold = (same_fo & ~same_sf).astype("int32")
    count_super = (same_fo & same_sf & ~same_fa).astype("int32")
    count_family = (same_fo & same_sf & same_fa).astype("int32")
    before_fp = 1 - work["seen_fp"]
    work["fold_tp"] = count_fold * before_fp
    work["super_tp"] = count_super * before_fp
    work["family_tp"] = count_family * before_fp
    work["count_fold"] = count_fold
    work["count_super"] = count_super
    work["count_family"] = count_family

    agg = (
        work.groupby("qid", sort=False)
        .agg(
            focnt=("fold_tp", "sum"), fotot=("count_fold", "sum"),
            sfcnt=("super_tp", "sum"), sftot=("count_super", "sum"),
            facnt=("family_tp", "sum"), fatot=("count_family", "sum"),
        )
        .reset_index()
    )
    for tot in ("fotot", "sftot", "fatot"):
        agg[tot] = agg[tot].replace(0, 1)
    agg["fofrac"] = agg["focnt"] / agg["fotot"]
    agg["sfrac"] = agg["sfcnt"] / agg["sftot"]
    agg["fafrac"] = agg["facnt"] / agg["fatot"]

    out_prefix.parent.mkdir(parents=True, exist_ok=True)
    paths = {
        "fol": Path(str(out_prefix) + "_fol.tsv"),
        "sup": Path(str(out_prefix) + "_sup.tsv"),
        "fam": Path(str(out_prefix) + "_fam.tsv"),
    }
    agg[["qid", "focnt", "fotot", "fofrac"]].to_csv(paths["fol"], sep="\t", header=False, index=False)
    agg[["qid", "sfcnt", "sftot", "sfrac"]].to_csv(paths["sup"], sep="\t", header=False, index=False)
    agg[["qid", "facnt", "fatot", "fafrac"]].to_csv(paths["fam"], sep="\t", header=False, index=False)
    return paths


def calc_fp_rates_catalog(aln_tsv_path: Path, cla: pd.DataFrame, out_prefix: Path) -> dict[str, Path]:
    """Catalog protocol (bench.noselfhit.awk): denominator = all catalog homologs."""
    print(f"  catalog (bench.noselfhit.awk): {aln_tsv_path}", flush=True)
    id2fam = dict(zip(cla["id"].astype(str), cla["fa"].astype(str)))
    id2sfam = dict(zip(cla["id"].astype(str), cla["sf"].astype(str)))
    id2fold = dict(zip(cla["id"].astype(str), cla["fo"].astype(str)))
    fam_cnt: dict[str, int] = Counter(cla["fa"].astype(str))
    sfam_cnt: dict[str, int] = Counter(cla["sf"].astype(str))
    fold_cnt: dict[str, int] = Counter(cla["fo"].astype(str))
    found_fam: dict[str, int] = defaultdict(int)
    found_sfam: dict[str, int] = defaultdict(int)
    found_fold: dict[str, int] = defaultdict(int)
    found_fp: dict[str, int] = defaultdict(int)

    n_lines = 0
    with aln_tsv_path.open() as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            qid, tid = parts[0], parts[1]
            n_lines += 1
            if qid not in id2fam:
                continue
            if tid not in id2fam:
                found_fp[qid] += 1
                continue
            if qid == tid:
                continue
            q_fam, q_sfam, q_fold = id2fam[qid], id2sfam[qid], id2fold[qid]
            t_fam, t_sfam, t_fold = id2fam[tid], id2sfam[tid], id2fold[tid]
            if found_fp[qid] < 1 and q_fold != t_fold:
                found_fp[qid] += 1
                continue
            if found_fp[qid] < 1 and q_fam == t_fam:
                found_fam[qid] += 1
                continue
            if found_fp[qid] < 1 and q_fam != t_fam and q_sfam == t_sfam:
                found_sfam[qid] += 1
                continue
            if found_fp[qid] < 1 and q_fam != t_fam and q_sfam != t_sfam and q_fold == t_fold:
                found_fold[qid] += 1
                continue

    print(f"  rows: {n_lines:,}", flush=True)
    rows_fam: list[tuple[str, int, int, float]] = []
    rows_sup: list[tuple[str, int, int, float]] = []
    rows_fol: list[tuple[str, int, int, float]] = []
    n_valid = 0
    for qid, fam in id2fam.items():
        if not fam:
            continue
        sfam = id2sfam[qid]
        fold = id2fold[qid]
        n_fam = fam_cnt[fam]
        n_sfam = sfam_cnt[sfam]
        n_fold = fold_cnt[fold]
        if not (n_fam > 1 and n_sfam - n_fam > 0 and n_fold - n_sfam > 0):
            continue
        n_valid += 1
        fam_tot = n_fam - 1
        sfam_tot = n_sfam - (n_fam - 1)
        fold_tot = n_fold - (n_sfam - 1)
        rows_fam.append((qid, found_fam[qid], fam_tot, found_fam[qid] / fam_tot))
        rows_sup.append((qid, found_sfam[qid], sfam_tot, found_sfam[qid] / sfam_tot))
        rows_fol.append((qid, found_fold[qid], fold_tot, found_fold[qid] / fold_tot))

    print(f"  valid queries: {n_valid:,} / lookup {len(id2fam):,}", flush=True)
    out_prefix.parent.mkdir(parents=True, exist_ok=True)
    paths = {
        "fol": Path(str(out_prefix) + "_fol.tsv"),
        "sup": Path(str(out_prefix) + "_sup.tsv"),
        "fam": Path(str(out_prefix) + "_fam.tsv"),
    }

    def _write(path: Path, rows: list[tuple[str, int, int, float]]) -> None:
        with path.open("w") as out:
            for qid, cnt, tot, frac in rows:
                out.write(f"{qid}\t{cnt}\t{tot}\t{frac:.6f}\n")

    _write(paths["fam"], rows_fam)
    _write(paths["sup"], rows_sup)
    _write(paths["fol"], rows_fol)
    return paths


def mean_sensitivity(level_tsv: Path) -> float:
    vals = []
    with level_tsv.open() as handle:
        for line in handle:
            parts = line.strip().split()
            if len(parts) >= 4:
                vals.append(float(parts[3]))
    return sum(vals) / len(vals) if vals else 0.0


def evaluate_all(skip_existing: bool = True) -> pd.DataFrame:
    ensure_work_dirs()
    build_scop_lookup(skip_existing=skip_existing)
    cla = load_scop_levels()
    print(f"SCOP levels: {len(cla)}")
    auc_rows: list[dict] = []
    for level_name, level_key in (("Family", "fam"), ("Superfamily", "sup"), ("Fold", "fol")):
        row: dict[str, float | str] = {"search_mode": "easy", "level": level_name}
        for method in selected_methods():
            label, key, protocol = method["name"], method["key"], method["protocol"]
            tsv_path = aln_tsv(key)
            out_prefix = metric_prefix(key)
            fam_path = Path(str(out_prefix) + "_fam.tsv")
            level_path = Path(str(out_prefix) + f"_{level_key}.tsv")
            print(f"\n[easy] {label}  protocol={protocol}")
            if not tsv_path.is_file():
                print(f"  [missing] alignment: {tsv_path}")
                continue
            skip_this = (
                protocol != "catalog"
                and skip_existing
                and fam_path.is_file()
                and fam_path.stat().st_size > 0
            )
            if not skip_this:
                if level_name == "Family":
                    try:
                        if protocol == "catalog":
                            calc_fp_rates_catalog(tsv_path, cla, out_prefix)
                            print("  [ok] catalog wrote fam/sup/fol")
                        else:
                            calc_fp_rates(tsv_path, cla, out_prefix)
                            print("  [ok] hitlist wrote fam/sup/fol")
                    except Exception as exc:
                        print(f"  [missing] {exc}")
                        traceback.print_exc()
                        continue
                    finally:
                        gc.collect()
            elif level_name == "Family":
                print("  [skip] metrics exist")
            if level_path.is_file():
                row[label] = mean_sensitivity(level_path)
                print(f"  {level_name} AUC={row[label]:.4f}")
        auc_rows.append(row)

    df = pd.DataFrame(auc_rows)
    csv_path = METRICS_DIR / "auc_easy.csv"
    df.to_csv(csv_path, index=False)
    print(f"\n[ok] AUC CSV: {csv_path}")
    return df


auc_df = evaluate_all(skip_existing=SKIP_EXISTING)
display(auc_df)
print("AUCs from different protocols are not the same metric.")
print("Next: 3_figures.ipynb")


## Verify


In [ ]:
rows = []
for method in selected_methods():
    row = {
        "method": method["name"],
        "key": method["key"],
        "protocol": method["protocol"],
        "aln": aln_tsv(method["key"]).is_file(),
    }
    for level in ("fam", "sup", "fol"):
        path = Path(str(metric_prefix(method["key"])) + f"_{level}.tsv")
        row[level] = path.is_file()
        if path.is_file():
            row[f"auc_{level}"] = round(mean_sensitivity(path), 4)
    rows.append(row)
verify_df = pd.DataFrame(rows)
display(verify_df)
missing = [r["key"] for r in rows if not (r["aln"] and r["fam"] and r["sup"] and r["fol"])]
if missing:
    raise SystemExit(f"Incomplete methods: {missing}")
print("[ok] search + evaluate complete")


## Cleanup


In [ ]:
cleanup_tmp(also_work_tmp=True)
print("Cleanup done. Products remain under work/ and bin/.")
